# Residual MCTS highway evaluation on Colab

This notebook clones a selected Git ref, creates an isolated virtual environment, runs `eval_mcts.py`, and saves the complete evaluation log plus run parameters to Google Drive.

The notebook writes a runtime agent config so the MCTS prior model and search parameters are controlled from one cell. Ensure the selected Git ref contains the entry point, base config, and prior model. Use a distinct `RUN_NAME` for parallel sessions.

In [ ]:
# Mount Google Drive. Colab will prompt you to authorize access.
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Experiment parameters (edit this cell before running).
from datetime import datetime, timezone

REPO_URL = "https://github.com/thowell332/RQL-Comparison.git"  #@param {type:"string"}
REPO_REF = "main"  #@param {type:"string"}
SUPERVISOR_REPO_URL = "https://github.com/phbui/norm-supervised-highway.git"  #@param {type:"string"}
SUPERVISOR_REPO_REF = "main"  #@param {type:"string"}

ENV_NAME = "highway-ME-basic-AddRightReward-v0"  #@param {type:"string"}
BASE_AGENT_CONFIG = "configs/HighwayEnv/agents/MCTSAgent/baselineMEPreRL.json"  #@param {type:"string"}
PRIOR_MODEL_PATH = "logs/highway-ME-basic-v0_Example_Pretrained.zip"  #@param {type:"string"}
N_EPISODES = 10  #@param {type:"integer"}
SEED = 239  #@param {type:"integer"}
SAFE_DECIDE = False  #@param {type:"boolean"}

# MCTS fields normally supplied by the agent JSON config.
MCTS_SIMULATIONS = 150  # `episodes` in the config
MCTS_HORIZON = 6
MCTS_TEMPERATURE = None  # Keep None to preserve the base config/default.
USE_DQN_PRIOR = True

RUN_NAME = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")  #@param {type:"string"}
DRIVE_RESULTS_ROOT = "/content/drive/MyDrive/aaai2027/RQL-Comparison/eval_mcts"  #@param {type:"string"}

In [ ]:
# Clone the requested ref and create a Colab-compatible evaluation environment.
import shutil
import subprocess
from pathlib import Path

PROJECT_DIR = Path("/content/RQL-Comparison")
SUPERVISOR_DIR = Path("/content/norm-supervised-highway-dependency")
VENV_DIR = Path("/content/venvs/rql-comparison")

for path in (PROJECT_DIR, SUPERVISOR_DIR, VENV_DIR):
    if path.exists():
        shutil.rmtree(path)

subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", REPO_REF], check=True)
subprocess.run(["git", "clone", SUPERVISOR_REPO_URL, str(SUPERVISOR_DIR)], check=True)
subprocess.run(["git", "-C", str(SUPERVISOR_DIR), "checkout", SUPERVISOR_REPO_REF], check=True)
# Reuse Colab's CUDA-enabled PyTorch while isolating experiment-specific packages.
subprocess.run(["python3", "-m", "venv", "--system-site-packages", str(VENV_DIR)], check=True)
VENV_PYTHON = VENV_DIR / "bin/python"

subprocess.run([str(VENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip"], check=True)
# The repository-wide requirements pin Python-3.9-era training packages. These are
# the complete dependencies needed by the highway MCTS evaluation path on current Colab.
EVAL_PACKAGES = [
    "numpy<2", "gym==0.23.1", "scipy", "pandas", "cloudpickle",
    "matplotlib", "tensorboard", "tensorboardX", "pygame", "tqdm",
    "rich", "pyyaml",
]
subprocess.run([str(VENV_PYTHON), "-m", "pip", "install", *EVAL_PACKAGES], check=True)
# eval_mcts.py imports this package, but it is maintained in the companion repo.
subprocess.run(
    [str(VENV_PYTHON), "-m", "pip", "install", "--no-deps", "-e", str(SUPERVISOR_DIR)],
    check=True,
)

entry_point = PROJECT_DIR / "eval_mcts.py"
if not entry_point.is_file():
    raise FileNotFoundError(f"Selected Git ref does not contain {entry_point}")
print(f"Project: {PROJECT_DIR}")
print(f"Python:  {VENV_PYTHON}")

In [ ]:
# Build the runtime agent config, run eval_mcts.py, and save all output to Drive.
import json
import os

RUN_DIR = Path(DRIVE_RESULTS_ROOT).expanduser() / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

def resolve_path(value):
    path = Path(value).expanduser()
    return path if path.is_absolute() else PROJECT_DIR / path

base_config_path = resolve_path(BASE_AGENT_CONFIG)
prior_model_path = resolve_path(PRIOR_MODEL_PATH)
for required_path in (base_config_path, prior_model_path):
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required asset not found: {required_path}. Commit it to the selected Git ref or use an absolute Drive path."
        )

agent_config = json.loads(base_config_path.read_text())
agent_config["episodes"] = MCTS_SIMULATIONS
agent_config["horizon"] = MCTS_HORIZON
agent_config["DQN_prior"] = int(USE_DQN_PRIOR)
agent_config["model_path"] = str(prior_model_path)
if MCTS_TEMPERATURE is not None:
    agent_config["temperature"] = MCTS_TEMPERATURE

runtime_config_path = RUN_DIR / "agent_config.json"
runtime_config_path.write_text(json.dumps(agent_config, indent=2))
manifest = {
    "repo_url": REPO_URL,
    "repo_ref": REPO_REF,
    "supervisor_repo_url": SUPERVISOR_REPO_URL,
    "supervisor_repo_ref": SUPERVISOR_REPO_REF,
    "env_name": ENV_NAME,
    "base_agent_config": str(base_config_path),
    "prior_model_path": str(prior_model_path),
    "n_episodes": N_EPISODES,
    "seed": SEED,
    "safe_decide": SAFE_DECIDE,
    "mcts_simulations": MCTS_SIMULATIONS,
    "mcts_horizon": MCTS_HORIZON,
    "mcts_temperature": MCTS_TEMPERATURE,
    "use_dqn_prior": USE_DQN_PRIOR,
}
(RUN_DIR / "run_parameters.json").write_text(json.dumps(manifest, indent=2))

command = [
    str(VENV_PYTHON), str(PROJECT_DIR / "eval_mcts.py"),
    "--env-name", ENV_NAME,
    "--agent-config", str(runtime_config_path),
    "--n-episodes", str(N_EPISODES),
    "--seed", str(SEED),
]
if SAFE_DECIDE:
    command.append("--safe-decide")

print("$", " ".join(command))
with (RUN_DIR / "evaluation.log").open("w", buffering=1) as log:
    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

print(f"Results saved to: {RUN_DIR}")

In [ ]:
# Verify the artifacts persisted to Drive.
for artifact in sorted(path for path in RUN_DIR.rglob("*") if path.is_file()):
    print(f"{artifact.relative_to(RUN_DIR)}  ({artifact.stat().st_size:,} bytes)")